In [ ]:
%matplotlib inline

# Find a global optimum with a local algorithm

## Problem

Local optimisation algorithms converge to the nearest local optimum
and may miss the global one on multimodal problems.

## Solution

Use the `MultiStart` algorithm, which generates several starting points
via a DOE algorithm and runs a local optimisation from each one.
The best local optimum across all starting points is returned as the result.

## Step-by-step guide


In [ ]:
from __future__ import annotations

from gemseo import execute_post
from gemseo.algos.design_space import DesignSpace
from gemseo.algos.doe.pydoe.settings.pydoe_fullfact import PYDOE_FULLFACT_Settings
from gemseo.algos.opt.multi_start.settings.multi_start_settings import (
    MultiStart_Settings,
)
from gemseo.algos.opt.scipy_local.settings.slsqp import SLSQP_Settings
from gemseo.disciplines.analytic import AnalyticDiscipline
from gemseo.post import BasicHistory_Settings
from gemseo.scenarios.mdo import MDOScenario

### 1. Build the multimodal problem

We define a constrained problem with a multimodal objective $x^3 - x + 1$:



In [ ]:
objective = AnalyticDiscipline({"obj": "x**3-x+1"})
constraint = AnalyticDiscipline({"cstr": "x**2+obj**2-1.5"})

design_space = DesignSpace()
design_space.add_variable("x", lower_bound=-1.5, upper_bound=1.5, value=1.5)

scenario = MDOScenario([objective, constraint], design_space)
scenario.add_objective("obj")
scenario.add_constraint("cstr", constraint_type="ineq")

### 2. Execute with the MultiStart algorithm

Combine a local algorithm (SLSQP) with a DOE to generate starting points
(here, a 10-point full-factorial design).
Set `multistart_file_path` to save the history of local optima:



In [ ]:
scenario.execute(
    MultiStart_Settings(
        max_iter=100,
        opt_algo_settings=SLSQP_Settings(),
        doe_algo_settings=PYDOE_FULLFACT_Settings(n_samples=10),
        multistart_file_path="multistart.hdf5",
    )
)

### 3. Inspect the results

Plot the full history (all sub-optimisation iterations concatenated):



In [ ]:
execute_post(
    scenario, BasicHistory_Settings(variable_names=["obj"], save=False, show=True)
)

Or plot only the local optima (one per starting point)
from the multistart history file:



In [ ]:
execute_post(
    "multistart.hdf5",
    BasicHistory_Settings(variable_names=["obj"], save=False, show=True),
)

## Summary

The `MultiStart` algorithm wraps any local optimisation algorithm
and runs it from multiple starting points generated by a DOE.
Use `multistart_file_path` to save the local optima history
and inspect it separately from the full optimisation history.

